# 03 - Transformer 架构


Transformer 把注意力、前馈网络、残差连接、LayerNorm 和位置/Token 嵌入组合成可堆叠的通用架构。本 notebook 会从单个 Transformer Block 开始，构建一个迷你 GPT，并训练它完成下一个 token 预测。

学习目标：

- 理解 Encoder-only、Decoder-only、Encoder-Decoder 的区别。
- 看懂 Transformer Block 内部的数据流。
- 理解 GPT 的 token embedding、position embedding 和输出投影。
- 掌握 next-token prediction 的训练方式。
- 理解自回归生成为什么一次生成一个 token。

## 环境准备与导入

这一段保留原教程的导入、标题打印和基础设置。先运行它，后续代码单元会复用这里导入的库、函数和随机种子。

In [1]:
"""
第五章 5.3：Transformer 架构
============================

Transformer 是现代 AI 的基础架构。
GPT、BERT、LLaMA、ChatGPT 等全部基于 Transformer。

"Attention Is All You Need" (2017) - 最重要的 AI 论文之一

本节内容：
1. Transformer 整体架构
2. 从零实现 Transformer Block
3. GPT 风格的解码器
4. 简单的文本生成
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

print("=" * 60)
print("第五章 5.3：Transformer 架构")
print("=" * 60)

第五章 5.3：Transformer 架构


## 1. Transformer 整体架构


Transformer 的关键不是某一个层，而是一组可以稳定堆叠的设计：注意力负责 token 间交互，FFN 负责每个位置的非线性变换，残差连接和 LayerNorm 稳定深层训练。

现代 LLM 大多采用 decoder-only 架构，因为它天然适合从左到右生成文本。

In [2]:
print("\n" + "=" * 60)
print("1. Transformer 整体架构")
print("=" * 60)

print("""
【Transformer 的两种变体】

1. Encoder-only (如 BERT):
   - 双向注意力（可以看到完整上下文）
   - 适合理解任务（分类、抽取等）

2. Decoder-only (如 GPT):
   - 因果注意力（只能看到左边的内容）
   - 适合生成任务（写文章、对话等）
   - 现在最流行的 LLM 架构！

3. Encoder-Decoder (如原始 Transformer):
   - 编码器处理输入，解码器生成输出
   - 适合翻译等序列到序列任务

【单个 Transformer Block 的结构】(Decoder 版本)

  输入
   ↓
  [多头自注意力 (Masked)]  ← 核心！
   ↓  + 残差连接
  [Layer Normalization]
   ↓
  [前馈网络 (FFN)]        ← 两层全连接
   ↓  + 残差连接
  [Layer Normalization]
   ↓
  输出

然后堆叠 N 个这样的 Block（GPT-3 有 96 层！）

【关键组件】
- 多头注意力: 学习词之间的关系
- FFN: 对每个位置独立地非线性变换
- 残差连接: 缓解深层网络的梯度问题
- Layer Norm: 稳定训练
""")


1. Transformer 整体架构

【Transformer 的两种变体】

1. Encoder-only (如 BERT):
   - 双向注意力（可以看到完整上下文）
   - 适合理解任务（分类、抽取等）

2. Decoder-only (如 GPT):
   - 因果注意力（只能看到左边的内容）
   - 适合生成任务（写文章、对话等）
   - 现在最流行的 LLM 架构！

3. Encoder-Decoder (如原始 Transformer):
   - 编码器处理输入，解码器生成输出
   - 适合翻译等序列到序列任务

【单个 Transformer Block 的结构】(Decoder 版本)

  输入
   ↓
  [多头自注意力 (Masked)]  ← 核心！
   ↓  + 残差连接
  [Layer Normalization]
   ↓
  [前馈网络 (FFN)]        ← 两层全连接
   ↓  + 残差连接
  [Layer Normalization]
   ↓
  输出

然后堆叠 N 个这样的 Block（GPT-3 有 96 层！）

【关键组件】
- 多头注意力: 学习词之间的关系
- FFN: 对每个位置独立地非线性变换
- 残差连接: 缓解深层网络的梯度问题
- Layer Norm: 稳定训练



## 2. 从零实现各组件


这一节实现的是 GPT 风格 Block。注意这里使用的是 pre-norm 结构：先 LayerNorm，再进入 Attention 或 FFN，最后加残差。

pre-norm 在深层 Transformer 中通常更稳定，因为梯度可以更顺畅地沿残差路径传播。

In [3]:
print("\n" + "=" * 60)
print("2. 从零实现 Transformer 组件")
print("=" * 60)

class MultiHeadAttention(nn.Module):
    """多头注意力"""
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        batch, seq_len, _ = x.shape
        
        Q = self.W_Q(x).view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, V)
        
        context = context.transpose(1, 2).contiguous().view(batch, seq_len, self.d_model)
        return self.W_O(context)


class FeedForward(nn.Module):
    """前馈网络 (Position-wise FFN)"""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.activation = nn.GELU()  # GPT 使用 GELU 而不是 ReLU
    
    def forward(self, x):
        return self.linear2(self.activation(self.linear1(x)))


class TransformerBlock(nn.Module):
    """单个 Transformer Block (Decoder 风格)"""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # 自注意力 + 残差连接 + LayerNorm
        attn_output = self.attention(self.norm1(x), mask)
        x = x + self.dropout(attn_output)
        
        # FFN + 残差连接 + LayerNorm
        ffn_output = self.ffn(self.norm2(x))
        x = x + self.dropout(ffn_output)
        
        return x

# 测试
print("--- 测试 TransformerBlock ---")
d_model = 64
n_heads = 4
d_ff = 256

block = TransformerBlock(d_model, n_heads, d_ff)
x = torch.randn(2, 10, d_model)  # 2句话, 10个词, 64维

# 因果 mask
mask = torch.tril(torch.ones(10, 10)).unsqueeze(0).unsqueeze(0)
output = block(x, mask)

print(f"输入: {list(x.shape)}")
print(f"输出: {list(output.shape)}")
print(f"✓ TransformerBlock 正常工作")

# 验证残差连接（输出和输入维度相同）
assert x.shape == output.shape
print("✓ 输入输出形状一致（残差连接）")


2. 从零实现 Transformer 组件
--- 测试 TransformerBlock ---
输入: [2, 10, 64]
输出: [2, 10, 64]
✓ TransformerBlock 正常工作
✓ 输入输出形状一致（残差连接）


## 3. 完整的 GPT 模型


GPT 的输入不是单纯 token embedding，还要加 position embedding。因为模型需要同时知道“词是什么”和“处于哪个位置”。

输出 logits 的形状是 `(batch, seq_len, vocab_size)`，表示每个位置都在预测下一个 token 的分布。

In [4]:
print("\n" + "=" * 60)
print("3. 完整的 GPT 风格模型")
print("=" * 60)

class MiniGPT(nn.Module):
    """
    迷你 GPT 模型
    
    完整结构:
    Token Embedding + Position Embedding
    → N × TransformerBlock
    → LayerNorm
    → Linear (投影回词表大小)
    """
    
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        
        # 嵌入层
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Transformer 层（堆叠 N 个）
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        
        # 输出
        self.norm = nn.LayerNorm(d_model)
        self.output_projection = nn.Linear(d_model, vocab_size)
    
    def forward(self, input_ids):
        batch, seq_len = input_ids.shape
        
        # 嵌入
        token_emb = self.token_embedding(input_ids)
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        pos_emb = self.position_embedding(positions)
        x = self.dropout(token_emb + pos_emb)
        
        # 因果 mask
        mask = torch.tril(torch.ones(seq_len, seq_len, device=input_ids.device))
        mask = mask.unsqueeze(0).unsqueeze(0)
        
        # 通过所有 Transformer Block
        for block in self.blocks:
            x = block(x, mask)
        
        # 输出 logits
        x = self.norm(x)
        logits = self.output_projection(x)  # (batch, seq_len, vocab_size)
        
        return logits
    
    def generate(self, input_ids, max_new_tokens, temperature=1.0):
        """自回归生成"""
        for _ in range(max_new_tokens):
            # 截断到最大序列长度
            x = input_ids[:, -self.max_seq_len:]
            
            # 前向传播
            logits = self.forward(x)
            
            # 只看最后一个位置的预测
            next_token_logits = logits[:, -1, :] / temperature
            
            # 采样（或 argmax）
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            # 拼接
            input_ids = torch.cat([input_ids, next_token], dim=1)
        
        return input_ids

# 创建迷你 GPT
print("""
创建一个迷你 GPT 模型:
  词表大小: 50
  嵌入维度: 64
  注意力头: 4
  层数: 3
  FFN 维度: 256
  最大序列长度: 32
""")

mini_gpt = MiniGPT(
    vocab_size=50,
    d_model=64,
    n_heads=4,
    n_layers=3,
    d_ff=256,
    max_seq_len=32
)

total_params = sum(p.numel() for p in mini_gpt.parameters())
print(f"模型参数量: {total_params:,}")
print(f"\n对比真实模型参数量:")
print(f"  GPT-2:    117M (1.17亿)")
print(f"  GPT-3:    175B (1750亿)")
print(f"  LLaMA-7B: 7B  (70亿)")
print(f"  我们的:   {total_params:,} (约{total_params//1000}K)")

# 测试前向传播
input_ids = torch.randint(0, 50, (2, 10))  # 2句话, 每句10个token
logits = mini_gpt(input_ids)
print(f"\n前向传播测试:")
print(f"  输入: {list(input_ids.shape)}")
print(f"  输出 logits: {list(logits.shape)}")
assert logits.shape == (2, 10, 50)
print("✓ 模型结构正确")


3. 完整的 GPT 风格模型

创建一个迷你 GPT 模型:
  词表大小: 50
  嵌入维度: 64
  注意力头: 4
  层数: 3
  FFN 维度: 256
  最大序列长度: 32

模型参数量: 158,578

对比真实模型参数量:
  GPT-2:    117M (1.17亿)
  GPT-3:    175B (1750亿)
  LLaMA-7B: 7B  (70亿)
  我们的:   158,578 (约158K)

前向传播测试:
  输入: [2, 10]
  输出 logits: [2, 10, 50]
✓ 模型结构正确


## 4. 训练迷你 GPT


训练语言模型时，输入和目标只差一个位置：输入是前面的 token，目标是右移一位后的 token。这就是 next-token prediction。

虽然这里的数据只是重复模式，但训练流程和真实 GPT 完全同构：前向传播、交叉熵损失、反向传播、参数更新。

In [5]:
print("\n" + "=" * 60)
print("4. 训练迷你 GPT (字符级语言模型)")
print("=" * 60)

print("""
【任务】
训练模型学会一个简单的模式：重复序列
输入: [1, 2, 3, 4, 5] → 预测下一个: [2, 3, 4, 5, 1]

这就是"下一个 token 预测"(Next Token Prediction)
= GPT 的核心训练目标！
""")

# 生成简单的训练数据（重复模式）
def generate_repeat_data(n_samples=500, seq_len=8, vocab_size=10):
    """生成重复模式的数据"""
    data = []
    for _ in range(n_samples):
        # 随机选一个短序列，然后重复
        pattern_len = np.random.randint(2, 5)
        pattern = np.random.randint(1, vocab_size, pattern_len)
        # 重复 pattern 填满 seq_len
        full_seq = np.tile(pattern, seq_len // pattern_len + 1)[:seq_len + 1]
        data.append(full_seq)
    return np.array(data)

np.random.seed(42)
data = generate_repeat_data(n_samples=1000, seq_len=16, vocab_size=10)
print(f"训练数据示例:")
for i in range(3):
    print(f"  输入: {data[i][:-1].tolist()}")
    print(f"  目标: {data[i][1:].tolist()}")
    print()

# 准备数据
train_data = torch.LongTensor(data[:800])
test_data = torch.LongTensor(data[800:])

# 创建较小的模型
small_gpt = MiniGPT(
    vocab_size=10,
    d_model=32,
    n_heads=4,
    n_layers=2,
    d_ff=128,
    max_seq_len=16
)

optimizer = optim.Adam(small_gpt.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# 训练
print("--- 开始训练 ---")
n_epochs = 30
batch_size = 64

for epoch in range(n_epochs):
    small_gpt.train()
    total_loss = 0
    
    indices = np.random.permutation(len(train_data))
    for i in range(0, len(train_data), batch_size):
        batch_idx = indices[i:i+batch_size]
        batch = train_data[batch_idx]
        
        input_ids = batch[:, :-1]   # 输入: 前15个token
        targets = batch[:, 1:]       # 目标: 后15个token（右移一位）
        
        # 前向传播
        logits = small_gpt(input_ids)
        
        # 计算损失
        loss = criterion(logits.view(-1, 10), targets.reshape(-1))
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    if epoch % 5 == 0 or epoch == n_epochs - 1:
        # 评估
        small_gpt.eval()
        with torch.no_grad():
            test_input = test_data[:, :-1]
            test_target = test_data[:, 1:]
            test_logits = small_gpt(test_input)
            test_loss = criterion(test_logits.view(-1, 10), test_target.reshape(-1))
            
            # 准确率
            pred = test_logits.argmax(dim=-1)
            acc = (pred == test_target).float().mean()
        
        print(f"  Epoch {epoch:2d}: train_loss={total_loss/(len(train_data)//batch_size):.4f}, "
              f"test_loss={test_loss:.4f}, accuracy={acc:.4f}")

# 生成测试
print("\n--- 文本生成测试 ---")
small_gpt.eval()
# 给定前几个 token，让模型续写
prompts = [[1, 2, 3], [5, 6, 7], [3, 4]]

for prompt in prompts:
    input_ids = torch.LongTensor([prompt])
    generated = small_gpt.generate(input_ids, max_new_tokens=8, temperature=0.5)
    print(f"  输入: {prompt} → 生成: {generated[0].tolist()}")

print("\n(模型应该学会了重复模式)")


4. 训练迷你 GPT (字符级语言模型)

【任务】
训练模型学会一个简单的模式：重复序列
输入: [1, 2, 3, 4, 5] → 预测下一个: [2, 3, 4, 5, 1]

这就是"下一个 token 预测"(Next Token Prediction)
= GPT 的核心训练目标！

训练数据示例:
  输入: [4, 8, 5, 7, 4, 8, 5, 7, 4, 8, 5, 7, 4, 8, 5, 7]
  目标: [8, 5, 7, 4, 8, 5, 7, 4, 8, 5, 7, 4, 8, 5, 7, 4]

  输入: [3, 7, 8, 3, 7, 8, 3, 7, 8, 3, 7, 8, 3, 7, 8, 3]
  目标: [7, 8, 3, 7, 8, 3, 7, 8, 3, 7, 8, 3, 7, 8, 3, 7]

  输入: [4, 8, 4, 8, 4, 8, 4, 8, 4, 8, 4, 8, 4, 8, 4, 8]
  目标: [8, 4, 8, 4, 8, 4, 8, 4, 8, 4, 8, 4, 8, 4, 8, 4]

--- 开始训练 ---
  Epoch  0: train_loss=2.5367, test_loss=2.2129, accuracy=0.1637
  Epoch  5: train_loss=1.5812, test_loss=1.2644, accuracy=0.6203
  Epoch 10: train_loss=0.9876, test_loss=0.7573, accuracy=0.7756
  Epoch 15: train_loss=0.7084, test_loss=0.5295, accuracy=0.8431
  Epoch 20: train_loss=0.5921, test_loss=0.4403, accuracy=0.8675
  Epoch 25: train_loss=0.5264, test_loss=0.4109, accuracy=0.8766
  Epoch 29: train_loss=0.4874, test_loss=0.3999, accuracy=0.8791

--- 文本生成测试 ---
  输入: [1, 2, 3] → 生成: [1

## 5. 理解 GPT 的训练


大语言模型的训练目标看起来简单，但为了持续预测下一个 token，模型必须学习语法、事实、风格、指代、推理和世界知识。

规模化训练让这个简单目标产生强大能力，但架构和损失函数的核心仍然是本 notebook 中的迷你 GPT。

In [6]:
print("\n" + "=" * 60)
print("5. 理解大语言模型的训练")
print("=" * 60)

print("""
【GPT 的训练本质上就是我们刚才做的事情，只是规模大得多】

我们的模型:
  数据: 1000个简单重复序列
  参数: ~10K
  训练: 30 epochs, 几秒钟

GPT-3:
  数据: 整个互联网文本 (45TB)
  参数: 175B (1750亿)
  训练: 数千个 GPU 训练数月
  成本: 约 460 万美元

但核心思想完全一样：
  输入: "The cat sat on the"
  目标: "cat sat on the mat"
  损失: CrossEntropy(预测的下一个词, 真实的下一个词)

【为什么这么简单的目标能产生"智能"？】

预测下一个词迫使模型理解：
  - 语法: "He ___(goes/go) to school" → 需要理解主语单复数
  - 事实: "The capital of France is ___" → 需要"知道"地理知识
  - 推理: "If A>B and B>C, then A___(>/<)C" → 需要逻辑推理
  - 情感: "I'm so happy because ___" → 需要理解因果和情感

通过在海量文本上预测下一个词，模型被迫学到了语言的各种规律。

【关键技术突破】
1. 规模: 更大的模型 + 更多的数据 = 更好的效果 (Scaling Law)
2. Transformer: 可以高效并行训练（不像 RNN）
3. 对齐: RLHF (人类反馈强化学习) → 让模型输出更有帮助
""")

print("\n" + "=" * 60)
print("本节总结")
print("=" * 60)
print("""
关键要点：
1. Transformer Block = Multi-Head Attention + FFN + 残差 + LayerNorm
2. GPT = Token Embedding + Position Embedding + N × TransformerBlock + 输出投影
3. 训练目标: 给定前文，预测下一个 token (Next Token Prediction)
4. 生成方式: 自回归 — 一次生成一个 token，把生成的接到输入后面继续
5. 大语言模型的"智能"来自于在海量数据上做下一个词预测

我们的迷你 GPT 展示了完整的架构，
真实的 GPT 只是规模更大（更多层、更多头、更大维度、更多数据）

下一节：使用预训练模型 → Hugging Face 实践
""")


5. 理解大语言模型的训练

【GPT 的训练本质上就是我们刚才做的事情，只是规模大得多】

我们的模型:
  数据: 1000个简单重复序列
  参数: ~10K
  训练: 30 epochs, 几秒钟

GPT-3:
  数据: 整个互联网文本 (45TB)
  参数: 175B (1750亿)
  训练: 数千个 GPU 训练数月
  成本: 约 460 万美元

但核心思想完全一样：
  输入: "The cat sat on the"
  目标: "cat sat on the mat"
  损失: CrossEntropy(预测的下一个词, 真实的下一个词)

【为什么这么简单的目标能产生"智能"？】

预测下一个词迫使模型理解：
  - 语法: "He ___(goes/go) to school" → 需要理解主语单复数
  - 事实: "The capital of France is ___" → 需要"知道"地理知识
  - 推理: "If A>B and B>C, then A___(>/<)C" → 需要逻辑推理
  - 情感: "I'm so happy because ___" → 需要理解因果和情感

通过在海量文本上预测下一个词，模型被迫学到了语言的各种规律。

【关键技术突破】
1. 规模: 更大的模型 + 更多的数据 = 更好的效果 (Scaling Law)
2. Transformer: 可以高效并行训练（不像 RNN）
3. 对齐: RLHF (人类反馈强化学习) → 让模型输出更有帮助


本节总结

关键要点：
1. Transformer Block = Multi-Head Attention + FFN + 残差 + LayerNorm
2. GPT = Token Embedding + Position Embedding + N × TransformerBlock + 输出投影
3. 训练目标: 给定前文，预测下一个 token (Next Token Prediction)
4. 生成方式: 自回归 — 一次生成一个 token，把生成的接到输入后面继续
5. 大语言模型的"智能"来自于在海量数据上做下一个词预测

我们的迷你 GPT 展示了完整的架构，
真实的 GPT 只是规模更大（更多层、更多头

## 学习检查：Transformer 应该掌握什么

你应该能回答：

- Transformer Block 中 Attention 和 FFN 各自负责什么？
- 残差连接为什么要求输入输出维度一致？
- GPT 为什么使用因果注意力？
- 语言模型训练时，输入和目标为什么是右移关系？
- `generate` 中为什么每次只取最后一个位置的 logits？

## 常见误区

1. **Transformer 不等于 GPT**：GPT 是 decoder-only Transformer 的一种应用。
2. **Attention 不负责所有计算**：FFN 占据大量参数，也提供重要非线性变换。
3. **logits 不是概率**：需要经过 softmax 才是概率分布。
4. **训练和生成不同**：训练可并行预测所有位置；生成必须自回归逐步采样。

## 深入理解：Transformer Block 的职责分工

一个 Transformer Block 可以看成两步：先让 token 之间交流，再让每个 token 独立思考。

多头自注意力负责信息交换。每个 token 根据注意力权重读取其它 token 的 Value，把上下文信息混合进自己的表示。FFN 则对每个位置独立应用同一个小型 MLP，增强非线性表达能力。

残差连接提供稳定路径：即使某个子层暂时学得不好，输入信息也能绕过它继续向后传播。LayerNorm 则控制数值分布，降低深层网络训练时的震荡。

## 训练和生成为什么不同

训练语言模型时，可以并行计算整段序列中每个位置的预测。比如输入 `[1,2,3,4]`，模型同时预测 `[2,3,4,下一个]`。这是因为训练数据中的真实答案已经存在。

生成时没有未来答案。模型只能先根据已有上下文预测一个 token，把这个 token 接到输入后面，再预测下一个。这叫自回归生成。

所以 Transformer 的训练可以高度并行，而生成通常是逐 token 的。这也是为什么大模型推理优化非常关注 KV cache：它可以缓存已经计算过的 Key/Value，避免每生成一个 token 都重复计算全部历史。

## 从 MiniGPT 到真实 GPT

本 notebook 中的 MiniGPT 和真实 GPT 在结构上是同一类模型：token embedding、position embedding、多层 decoder block、输出投影、next-token loss。

真实模型的差异主要在规模和工程细节：更多层、更大隐藏维度、更多注意力头、更长上下文、更大词表、更多数据、更稳定的训练策略，以及分布式训练和推理优化。

因此，不要因为 MiniGPT 很小就忽视它。只要你能解释 MiniGPT 的 forward、loss 和 generate，就已经掌握了理解大语言模型架构的主线。

## 核心术语对照

- **Token Embedding**：把 token id 转成向量。
- **Position Embedding**：补充位置信息，让模型知道 token 顺序。
- **Transformer Block**：由自注意力、FFN、残差连接和 LayerNorm 组成的可堆叠模块。
- **Logits**：模型输出的未归一化分数，经过 softmax 才是概率。
- **Next Token Prediction**：给定前文预测下一个 token，是 GPT 的核心训练目标。
- **Autoregressive Generation**：自回归生成，每次生成一个 token，再接回输入继续生成。

把这些术语连起来，就是 GPT 的主流程：token id 进入 embedding，经过多层 block，输出 logits，用 next-token loss 训练，用自回归方式生成。

## 课后练习：改造 MiniGPT

建议你尝试三个实验：第一，把 Transformer 层数从 2 改成 1 或 4，观察参数量和训练效果变化；第二，把 `temperature` 改成 0.2、1.0、1.5，观察生成序列的稳定性和随机性；第三，换一种简单数据模式，例如递增数字或交替数字，看模型是否还能学会。

这些练习能帮助你理解模型容量、训练数据和采样策略之间的关系。真实 LLM 也是同样的问题，只是规模大得多。

## 概念对照：BERT、GPT 与原始 Transformer

原始 Transformer 是 encoder-decoder 架构，最初用于机器翻译。Encoder 读完整输入句子，Decoder 自回归生成输出句子，并通过 cross-attention 读取 encoder 表示。

BERT 主要使用 encoder-only 架构。它能双向看上下文，因此适合理解任务，例如文本分类、命名实体识别、阅读理解和向量检索。

GPT 主要使用 decoder-only 架构。它使用因果注意力，只能看左侧上下文，因此适合生成任务，例如续写、对话、代码生成和工具调用。

这三类模型共享很多组件，但 mask、训练目标和使用方式不同。不要只记模型名字，要看它允许看到哪些上下文，以及它被训练来做什么。

## 读懂 Transformer 代码的顺序

建议按数据流阅读：

1. 输入 token id 的形状是 `(batch, seq_len)`。
2. token embedding 和 position embedding 相加，得到 `(batch, seq_len, d_model)`。
3. 每个 TransformerBlock 保持形状不变。
4. 输出投影把最后一维从 `d_model` 变成 `vocab_size`。
5. loss 把 logits 展平成二维，把 targets 展平成一维，计算交叉熵。

读代码时先抓住 shape，再看每个模块内部细节。Transformer 很复杂，但它最核心的工程约束很简单：block 输入输出维度必须一致，才能反复堆叠。